In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field 
import operator

In [2]:
model = ChatOllama(model='Phi3')

In [3]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)


In [4]:
structured_model = model.with_structured_output(EvaluationSchema)


In [5]:
essay = """ The Role of India in Artificial Intelligence: A Rising Global ForceIntroductionArtificial Intelligence is no longer a distant technological frontier — it is the defining force reshaping economies, governance, healthcare, education, and national security in the twenty-first century. Among the nations positioning themselves at the forefront of this transformation, India stands out as a uniquely compelling story. With its vast pool of technical talent, a rapidly expanding digital infrastructure, a young and ambitious population, and a government increasingly committed to AI-led development, India is transitioning from being a participant in the global AI conversation to becoming one of its most influential architects. The role India plays in artificial intelligence today — and the role it aspires to play tomorrow — carries enormous consequences not just for its 1.4 billion citizens, but for the entire world.India's AI Talent: The Human FoundationAt the heart of India's AI ambitions lies its most formidable asset: human capital. India produces over 1.5 million engineering graduates every year, a significant portion of whom specialize in computer science, data science, and related disciplines. Indian professionals occupy leadership positions in the world's most powerful technology companies — from Google's Sundar Pichai to Microsoft's Satya Nadella — and Indian-origin researchers populate the top AI laboratories at MIT, Stanford, Carnegie Mellon, and DeepMind.Within India itself, premier institutions like the Indian Institutes of Technology (IITs) and the Indian Institute of Science (IISc) have established dedicated AI and machine learning research centers. The country's large English-speaking technical workforce makes it an attractive destination for global AI research collaboration and outsourcing. India is not merely training workers for AI — it is cultivating thinkers, researchers, and innovators who are actively pushing the boundaries of what AI can do.Government Policy and the National AI MissionIndia's government has recognized that AI leadership is not accidental — it must be deliberately built. In 2018, NITI Aayog, India's national policy think tank, released a landmark strategy document titled "National Strategy for Artificial Intelligence," which framed AI as a tool for inclusive growth and positioned India to become the "AI garage" of the world — a nation that develops AI solutions not only for itself but for other developing nations facing similar challenges.Building on this foundation, the government launched the IndiaAI Mission in 2024 with an outlay of over ₹10,000 crore (approximately 1.2 billion USD). The mission focuses on five pillars: building AI compute infrastructure, developing indigenous large language models and datasets, creating AI application ecosystems, nurturing AI startups, and skilling the workforce. The establishment of the IndiaAI compute platform — aimed at providing affordable GPU access to researchers and startups — addresses one of the most critical bottlenecks facing AI development in the country.India has also taken a principled stance on AI governance, advocating for what it calls "AI for all" — a framework that emphasizes equitable access, ethical development, and the application of AI to solve real human problems rather than merely maximizing profit. This philosophy was prominently echoed during India's G20 presidency in 2023, where AI governance featured as a central agenda item.AI for Social Good: India's Unique OpportunityWhat makes India's AI story distinctly compelling is the scale and nature of the problems it can solve. Unlike the AI use cases dominant in Western economies — advertising optimization, financial trading, autonomous vehicles — India faces challenges of a different magnitude: hundreds of millions living without adequate healthcare, massive agricultural inefficiencies, complex multilingual communication barriers, and a vast informal economy largely invisible to digital systems.In healthcare, AI is being deployed to detect tuberculosis and diabetic retinopathy in rural populations where specialist doctors are scarce. Startups like Niramai are using thermal imaging and AI to screen for breast cancer affordably and non-invasively. The government's Ayushman Bharat Digital Mission is building a health data infrastructure that could enable population-scale AI diagnostics.In agriculture, which employs nearly half of India's workforce, AI-powered tools are helping farmers predict weather patterns, detect crop diseases through smartphone images, and receive personalized advisories in local languages. Initiatives like the Kisan AI platform and partnerships with organizations like the International Crops Research Institute are beginning to transform how smallholder farmers make decisions.In education, AI tools are being developed to deliver personalized learning in India's 22 officially recognized languages and hundreds of dialects — a challenge that has no parallel anywhere in the world. If solved, India's multilingual AI models could become the template for linguistic inclusion globally.In governance, AI is streamlining public service delivery, detecting tax fraud, managing traffic in urban areas, and enabling predictive policing — though the latter also raises significant civil liberties concerns that India must navigate carefully. """

In [6]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'

structured_model.invoke(prompt).feedback

"The essay provides a compelling narrative about India's role in AI with clear examples of how it is harnessing its demographic and educational strength to spearhead innovation, research, policy-making, startups, and social good initiatives. It successfully outlines the Indian government's strategic approach towards fostering a conducive environment for AI development through mission statements like IndiaAI Mission."

In [7]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float
    

In [8]:
def evaluate_language(state: UPSCState):
    prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {
        'language_feedback': output.feedback,
        'individual_scores': [output.score]   
    }

In [9]:
def evaluate_analysis(state: UPSCState):
    prompt = f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {
        'analysis_feedback': output.feedback,
        'individual_scores': [output.score]   
    }

In [10]:
def evaluate_clarity(state: UPSCState):
    prompt = f"Evaluate the clarity of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {
        'clarity_feedback': output.feedback,
        'individual_scores': [output.score]  
    }

In [11]:
def evaluate_thought(state: UPSCState):
    prompt = f"Evaluate the quality of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {
        'overall_feedback': output.feedback,
        'individual_scores': [output.score]   
    }

In [12]:
def evaluate_evaluation(state: UPSCState):
    scores = state['individual_scores']
    avg = sum(scores) / len(scores) if scores else 0.0
    return {'avg_score': avg}

In [13]:
# --- Graph ---
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('evaluate_evaluation', evaluate_evaluation)

In [14]:
# Parallel evaluation nodes
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge(START, 'evaluate_clarity')

In [15]:
# All feed into final evaluation
graph.add_edge('evaluate_language', 'evaluate_evaluation')
graph.add_edge('evaluate_analysis', 'evaluate_evaluation')
graph.add_edge('evaluate_thought', 'evaluate_evaluation')
graph.add_edge('evaluate_clarity', 'evaluate_evaluation')

In [16]:
graph.add_edge('evaluate_evaluation', END)


In [17]:
workflow = graph.compile()


In [18]:
essay_text = """The Role of India in Artificial Intelligence: A Rising Global Force
Introduction
Artificial Intelligence is no longer a distant technological frontier — it is the defining force reshaping economies, governance, healthcare, education, and national security in the twenty-first century. Among the nations positioning themselves at the forefront of this transformation, India stands out as a uniquely compelling story. With its vast pool of technical talent, a rapidly expanding digital infrastructure, a young and ambitious population, and a government increasingly committed to AI-led development, India is transitioning from being a participant in the global AI conversation to becoming one of its most influential architects. The role India plays in artificial intelligence today — and the role it aspires to play tomorrow — carries enormous consequences not just for its 1.4 billion citizens, but for the entire world.
India's AI Talent: The Human Foundation
At the heart of India's AI ambitions lies its most formidable asset: human capital. India produces over 1.5 million engineering graduates every year, a significant portion of whom specialize in computer science, data science, and related disciplines. Indian professionals occupy leadership positions in the world's most powerful technology companies — from Google's Sundar Pichai to Microsoft's Satya Nadella — and Indian-origin researchers populate the top AI laboratories at MIT, Stanford, Carnegie Mellon, and DeepMind.
Within India itself, premier institutions like the Indian Institutes of Technology (IITs) and the Indian Institute of Science (IISc) have established dedicated AI and machine learning research centers. The country's large English-speaking technical workforce makes it an attractive destination for global AI research collaboration and outsourcing. India is not merely training workers for AI — it is cultivating thinkers, researchers, and innovators who are actively pushing the boundaries of what AI can do.
Government Policy and the National AI Mission
India's government has recognized that AI leadership is not accidental — it must be deliberately built. In 2018, NITI Aayog, India's national policy think tank, released a landmark strategy document titled "National Strategy for Artificial Intelligence," which framed AI as a tool for inclusive growth and positioned India to become the "AI garage" of the world — a nation that develops AI solutions not only for itself but for other developing nations facing similar challenges.
Building on this foundation, the government launched the IndiaAI Mission in 2024 with an outlay of over ₹10,000 crore (approximately 1.2 billion USD). The mission focuses on five pillars: building AI compute infrastructure, developing indigenous large language models and datasets, creating AI application ecosystems, nurturing AI startups, and skilling the workforce. The establishment of the IndiaAI compute platform — aimed at providing affordable GPU access to researchers and startups — addresses one of the most critical bottlenecks facing AI development in the country.
India has also taken a principled stance on AI governance, advocating for what it calls "AI for all" — a framework that emphasizes equitable access, ethical development, and the application of AI to solve real human problems rather than merely maximizing profit. This philosophy was prominently echoed during India's G20 presidency in 2023, where AI governance featured as a central agenda item.
AI for Social Good: India's Unique Opportunity
What makes India's AI story distinctly compelling is the scale and nature of the problems it can solve. Unlike the AI use cases dominant in Western economies — advertising optimization, financial trading, autonomous vehicles — India faces challenges of a different magnitude: hundreds of millions living without adequate healthcare, massive agricultural inefficiencies, complex multilingual communication barriers, and a vast informal economy largely invisible to digital systems.
In healthcare, AI is being deployed to detect tuberculosis and diabetic retinopathy in rural populations where specialist doctors are scarce. Startups like Niramai are using thermal imaging and AI to screen for breast cancer affordably and non-invasively. The government's Ayushman Bharat Digital Mission is building a health data infrastructure that could enable population-scale AI diagnostics.
In agriculture, which employs nearly half of India's workforce, AI-powered tools are helping farmers predict weather patterns, detect crop diseases through smartphone images, and receive personalized advisories in local languages. Initiatives like the Kisan AI platform and partnerships with organizations like the International Crops Research Institute are beginning to transform how smallholder farmers make decisions.
In education, AI tools are being developed to deliver personalized learning in India's 22 officially recognized languages and hundreds of dialects — a challenge that has no parallel anywhere in the world. If solved, India's multilingual AI models could become the template for linguistic inclusion globally.
In governance, AI is streamlining public service delivery, detecting tax fraud, managing traffic in urban areas, and enabling predictive policing — though the latter also raises significant civil liberties concerns that India must navigate carefully.
The Startup Ecosystem and Private Sector
India's AI startup ecosystem has exploded in the past decade. Bengaluru, Hyderabad, and Mumbai have emerged as AI innovation hubs, home to thousands of startups working across every sector. Companies like Sarvam AI are building India-specific large language models trained on Indian languages and contexts. Krutrim, founded by Ola's Bhavish Aggarwal, became India's first AI unicorn, signaling serious private capital flowing into indigenous AI development.
Global technology giants have also deepened their AI investments in India. Google, Microsoft, Amazon, and Meta have all established significant AI research and development centers in the country. India's combination of cost efficiency, talent depth, and market scale makes it an irresistible destination for AI investment. The growing venture capital interest — both domestic and foreign — in Indian AI startups suggests that the ecosystem is maturing rapidly from proof-of-concept to commercial scale.
Challenges India Must Overcome
India's AI journey is not without its obstacles. Several critical challenges threaten to slow its ascent if not addressed systematically.
Data quality and availability remain a major concern. While India generates enormous volumes of data, much of it is unstructured, poorly labeled, and not readily usable for AI training. The lack of high-quality datasets in Indian languages is a particular gap. Building curated, ethically sourced, publicly available datasets must be a national priority.
Compute infrastructure is another bottleneck. Training large AI models requires expensive GPU clusters that most Indian universities and startups cannot afford. While the IndiaAI Mission's compute platform is a step forward, India remains heavily dependent on foreign cloud providers and chip manufacturers — a vulnerability that geopolitical tensions around semiconductors make increasingly risky.
Ethical and regulatory frameworks are still nascent. Questions around algorithmic bias — especially in a society stratified by caste, religion, gender, and geography — are profound. An AI system trained primarily on urban, upper-caste data could systematically disadvantage marginalized communities. India needs robust, enforceable AI ethics standards before these systems are deployed at scale in high-stakes domains like policing, credit scoring, and hiring.
The digital divide also threatens to make AI a tool of exclusion rather than inclusion. Millions of Indians lack reliable internet access, smartphones, or digital literacy. If AI benefits flow primarily to the urban and educated, it will deepen existing inequalities rather than bridge them.
India's Role on the Global AI Stage
Beyond its domestic ambitions, India is increasingly shaping the global conversation about how AI should be governed and who it should serve. During its G20 presidency, India championed the idea that AI governance must reflect the interests of the Global South — countries that have largely been spectators in a conversation dominated by the United States, China, and Europe.
India's vision of "AI for humanity" — prioritizing applications that address poverty, disease, climate change, and food security — offers a counterpoint to the profit-driven and geopolitically competitive AI development models of the major powers. This moral and strategic positioning gives India influence disproportionate to its current technological capabilities.
India is also emerging as a trusted partner for AI collaboration with countries in Africa, Southeast Asia, and Latin America — nations that face development challenges similar to India's own and who look to India's technology solutions as more contextually relevant and affordable than those designed in Silicon Valley or Beijing.
Conclusion
India's role in artificial intelligence is still being written, but its trajectory is unmistakably upward. It possesses the talent, the scale, the democratic values, and the developmental urgency to become one of the world's most consequential AI powers. What distinguishes India's potential is not just its ability to build powerful AI systems, but its opportunity to demonstrate that AI can be a force for genuine human development — that intelligence, whether natural or artificial, is most valuable when it serves the many rather than the few.
The choices India makes in the next decade — about investment, regulation, ethics, inclusion, and international cooperation — will determine whether it fulfills this promise. If it does, India will not merely participate in the AI revolution. It will help define what that revolution is for."""

result = workflow.invoke({
    "essay": essay_text,
    "language_feedback": "",
    "analysis_feedback": "",
    "clarity_feedback": "",
    "overall_feedback": "",
    "individual_scores": [],
    "avg_score": 0.0
})

In [19]:
print("Language Feedback:", result['language_feedback'])
print("Analysis Feedback:", result['analysis_feedback'])
print("Clarity Feedback:", result['clarity_feedback'])
print("Overall Feedback:", result['overall_feedback'])
print("Individual Scores:", result['individual_scores'])
print("Average Score:", result['avg_score'])

Language Feedback: The essay presents a clear narrative about how India is becoming an influential player in the field of artificial intelligence (AI). The introduction sets up this claim effectively, drawing on facts such as AI-related venture capital and global recognition. It uses language aptly to convey urgency and significance.
Analysis Feedback: The essay presents a strong case for India's significant role and potential impact on global artificial intelligence development due to its rich human capital base, strategic government policies such as the National Strategy for Artificial Intelligence (2018) and the subsequent establishment of the IndiaAI Mission in 2024. The essay successfully highlighted how AI is being applied towards social good within various sectors unique to India's societal challenges, including healthcare accessibility with tools like Niramai for cancer screening using thermal imagery and innovative solutions aiding agricultural productivity in one of the world